# CONDOR–FlexDC Full-Parity Generic Inference, Optimization, Simulator Validation, and Comparison

This notebook is a version-neutral restoration of the original Model V3 inference and V3/V4 paired-comparison workflows. It clones both repositories, integrates generic sources without overwriting versioned files, supports every artifact acquisition mode, performs Predict One, shared-start optimization, multi-seed real FlexDC validation, fixed profile suites, custom rounds, resumable all-workload runs, multi-model comparisons, external reference comparisons, W&B logging, exact HTML output, standalone orchestration, recovery, and complete packaging. One model works without any paired-model requirement.

## 0. Environment, repositories, model artifacts, W&B, and execution controls

In [ ]:
from pathlib import Path
import importlib.util, os, platform

IS_COLAB = importlib.util.find_spec("google.colab") is not None
RUN_ENV = "colab" if IS_COLAB else "local_pc"
WORKSPACE = Path("/content/workspace") if IS_COLAB else Path.cwd() / "flexdc_inference_workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Repository controls retained from the original notebooks
# ---------------------------------------------------------------------------
COMDER_REPO_URL = "https://github.com/NetherMoon/CONDOR-FLEXDC.git"
FLEXDC_REPO_URL = "https://github.com/amenon871/FlexDC.git"
COMDER_BRANCH = "main"
FLEXDC_BRANCH = "main"
FORCE_FRESH_CLONE = False
UPDATE_EXISTING_REPOS = False
PRESERVE_LOCAL_REPO_CHANGES = True
CLONE_CONDOR_REPO = True
CLONE_FLEXDC_REPO = True
COMDER_ROOT_OVERRIDE = None
FLEXDC_ROOT_OVERRIDE = None
INSTALL_FLEXDC_REQUIREMENTS = False
INSTALL_FLEXDC_EDITABLE = True
INSTALL_GENERIC_SOURCES_IN_CONDOR = True
TORCH_CPU_THREADS = 4

# ---------------------------------------------------------------------------
# Generic package acquisition: existing_dir, path_zip, upload_zip
# ---------------------------------------------------------------------------
PACKAGE_MODE = "path_zip" if IS_COLAB else "existing_dir"
PACKAGE_ZIP_INPUT = Path("/content/FlexDC_Generic_Full_Parity_Bundle.zip") if IS_COLAB else None
PACKAGE_DIR_INPUT = None
AUTO_UPLOAD_PACKAGE = True
BUNDLE_ROOT_OVERRIDE = None

# ---------------------------------------------------------------------------
# Model artifact acquisition: existing, path_zip, upload_zip, multi_upload,
# repo_zip. One artifact works; multiple artifacts enable paired comparisons.
# ---------------------------------------------------------------------------
ARTIFACT_MODE = "multi_upload" if IS_COLAB else "existing"
MODEL_ARTIFACT_INPUTS = []
ARTIFACT_SEARCH_ROOT = None
AUTO_UPLOAD_MODEL_ARTIFACTS = True
PREFERRED_CHECKPOINT_ROLE = "best_feasibility"
EXPLICIT_CHECKPOINT_PATHS = []
MODEL_LABEL_OVERRIDES = []

# Optional prior output ZIP/directory. Recovered files are copied into the
# active output tree before any batch loop, matching the original recovery flow.
PRIOR_RESULTS_INPUT = None
AUTO_UPLOAD_PRIOR_RESULTS = False
RESUME_EXISTING_RESULTS = True

# ---------------------------------------------------------------------------
# W&B controls retained from the original inference notebook
# ---------------------------------------------------------------------------
USE_WANDB = True
WANDB_MODE = "online"                 # online, offline, disabled
WANDB_PROJECT = "flexdc-generic-inference"
WANDB_ENTITY = None
WANDB_RUN_ID_OVERRIDE = None
WANDB_FORCE_RELOGIN = False
AUTO_DOWNLOAD_FINAL_ZIP = True

# ---------------------------------------------------------------------------
# Scenario and starting point
# ---------------------------------------------------------------------------
SCENARIO = "J2-IT-ResNetInf-GPT2Train"   # any j2_pairwise stem or CUSTOM
CUSTOM_WORKLOAD_CONFIG = None
CUSTOM_EXPERIMENT_CONFIG = (
    "configs/experiment/new_iso/traditional_signal/generated_server_counts/"
    "exp_traditional_iso16_servers_1000.ini"
)
SERVER_COUNT = 1000
UTILIZATION = 0.80
SIMULATION_DURATION_OVERRIDE = None
START_POINT_MODE = "midpoint"         # midpoint, ratios, absolute
START_PBAR = None
START_R = None
START_PBAR_RATIO = 0.80
START_R_RATIO = 0.30
START_WEIGHTS = None

# Physical/bid overrides from the original optimizer controls
PBAR_LOWER_FACTOR = 0.9
PBAR_UPPER_FACTOR = 1.0
PR_UPPER_FACTOR = 1.2
R_LOWER_KW_PER_SERVER = 0.01
R_OVER_P_MAX = 0.6
PBAR_MIN_OVERRIDE = None
PBAR_MAX_OVERRIDE = None
R_MIN_OVERRIDE = None
R_MAX_OVERRIDE = None

# Weight domain controls. The explicit J=2 bounds are the study bounds; the
# generic relative controls remain available for other J values.
ENFORCE_FLEXDC_WEIGHT_BOUNDS = True
WEIGHT_MIN_FRACTION_OF_EQUAL = 0.1
WEIGHT_MAX_MULTIPLE_OF_EQUAL = 4.0
WEIGHT_MIN = 0.3
WEIGHT_MAX = 0.7

# ---------------------------------------------------------------------------
# Optimization controls retained from the original optimize-only script
# ---------------------------------------------------------------------------
RUN_PREDICT_ONE = True
RUN_OPTIMIZE_ONE = True
RUN_FLEXDC_VALIDATION = True
VALIDATE_STARTING_POINT = True
VALIDATE_TOP_K = True
SIMULATOR_SEEDS = [30, 31, 32]
VALIDATION_TIMEOUT_SECONDS = 1800
DRY_RUN_FLEXDC = False

OPTIMIZATION_MODE = "margin_constrained"  # pure_objective/exact_constrained/margin_constrained
MULTI_STARTS = 256
OPTIMIZATION_ITERATIONS = 1000
OPTIMIZATION_LR = 0.03
OPTIMIZATION_MIN_LR = 5e-4
TRACKING_PENALTY = 2000.0
QOS_PENALTY = 2000.0
PENALTY_RAMP_FRACTION = 0.30
TOP_K = 5
CANDIDATE_DISTANCE = 0.03
RANDOM_SEED = 101
NEAR_EQUAL_START_FRACTION = 0.25
HIGH_P_LOW_R_START_FRACTION = 0.25
LOG_EVERY = 25
TRACKING_MARGIN = 0.04
QOS_MARGIN = 0.01

# ---------------------------------------------------------------------------
# Full benchmark/orchestration controls retained from the original and paired
# notebooks. Every path uses the same generic model loader.
# ---------------------------------------------------------------------------
RUN_SAFETY_CALIBRATION = True
RUN_FIXED_PROFILE_SUITE = False
FIXED_SUITE_PER_BENCHMARK = 1
RUN_LEGACY_CUSTOM_ROUNDS = False
LEGACY_CUSTOM_ROUNDS = []
RUN_ALL_WORKLOADS = False
ALL_WORKLOAD_MAX_CASES = None
ALL_WORKLOAD_NAME_FILTER = None
ALL_WORKLOAD_CATEGORY_FILTER = None
ALL_WORKLOAD_VALIDATE_TOP_K = True
ALL_WORKLOAD_SEEDS = [30, 31, 32]
ALL_WORKLOAD_STOP_ON_ERROR = False
RUN_MULTI_MODEL_IDENTICAL_POINT_COMPARISON = True
RUN_MULTI_MODEL_SHARED_START_OPTIMIZATION = True
REFERENCE_POINTS_CSV = None            # optional Fatih/baseline/reference rows
RUN_FULL_ORCHESTRATOR_COMMAND = False
WRITE_HTML = True
DISPLAY_TABLES = True

print("Environment:", RUN_ENV)
print("Workspace:", WORKSPACE)
print("Scenario:", SCENARIO)
print("Artifact mode:", ARTIFACT_MODE)


## 1. Verify/install dependencies without replacing PyTorch

In [ ]:
import importlib, subprocess, sys
required={"numpy":"numpy","pandas":"pandas","scipy":"scipy","sklearn":"scikit-learn","matplotlib":"matplotlib","tabulate":"tabulate","tqdm":"tqdm","nbformat":"nbformat"}
if USE_WANDB and WANDB_MODE!="disabled": required["wandb"]="wandb"
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing: subprocess.run([sys.executable,"-m","pip","install","-q",*missing],check=True)
import torch
print("Python:",sys.version.split()[0]); print("Torch:",torch.__version__); print("CUDA:",torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))

## 2. Acquire the generic package

In [ ]:
import shutil, zipfile, sys

def upload_one(description):
    if not IS_COLAB:
        raise FileNotFoundError(description)
    from google.colab import files
    print("Upload", description)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f"Expected one file, got {list(uploaded)}")
    return (Path("/content") / next(iter(uploaded))).resolve()

if PACKAGE_MODE == "existing_dir":
    PACKAGE_ROOT = (
        Path(PACKAGE_DIR_INPUT).expanduser().resolve()
        if PACKAGE_DIR_INPUT is not None
        else Path.cwd().resolve()
    )
elif PACKAGE_MODE in {"path_zip", "upload_zip"}:
    package = Path(PACKAGE_ZIP_INPUT).expanduser().resolve() if PACKAGE_ZIP_INPUT is not None else None
    if (package is None or not package.exists()) and (PACKAGE_MODE == "upload_zip" or AUTO_UPLOAD_PACKAGE):
        package = upload_one("FlexDC_Generic_Full_Parity_Bundle.zip")
    if package is None or not package.exists():
        raise FileNotFoundError("Generic full-parity package ZIP")
    if not zipfile.is_zipfile(package):
        raise zipfile.BadZipFile(package)
    PACKAGE_ROOT = WORKSPACE / "generic_package"
    shutil.rmtree(PACKAGE_ROOT, ignore_errors=True)
    PACKAGE_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(package) as archive:
        bad = archive.testzip()
        if bad:
            raise zipfile.BadZipFile(f"CRC failure: {bad}")
        archive.extractall(PACKAGE_ROOT)
else:
    raise ValueError(f"Unknown PACKAGE_MODE: {PACKAGE_MODE}")

candidates = []
if BUNDLE_ROOT_OVERRIDE:
    candidates.append(Path(BUNDLE_ROOT_OVERRIDE))
candidates += [PACKAGE_ROOT, Path.cwd(), Path.cwd().parent]
BUNDLE_ROOT = None
for root in candidates:
    root = root.expanduser().resolve()
    if (root / "flexdc_generic_sources").exists():
        BUNDLE_ROOT = root
        break
    hits = list(root.rglob("flexdc_generic_sources")) if root.exists() else []
    if hits:
        BUNDLE_ROOT = hits[0].parent.resolve()
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError("flexdc_generic_sources")
SOURCE_DIR = BUNDLE_ROOT / "flexdc_generic_sources"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
from flexdc_colab_orchestration import *
print("Bundle root:", BUNDLE_ROOT)


## 3. Clone or update both repositories and install FlexDC

In [ ]:
COMDER_ROOT = Path(COMDER_ROOT_OVERRIDE).expanduser().resolve() if COMDER_ROOT_OVERRIDE else WORKSPACE / "comder-main"
FLEXDC_ROOT = Path(FLEXDC_ROOT_OVERRIDE).expanduser().resolve() if FLEXDC_ROOT_OVERRIDE else WORKSPACE / "FlexDC"

repo_states = []
if CLONE_CONDOR_REPO:
    repo_states.append(clone_or_update_repository(
        name="CONDOR-FLEXDC", url=COMDER_REPO_URL, destination=COMDER_ROOT,
        branch=COMDER_BRANCH, force_reclone=FORCE_FRESH_CLONE,
        update_existing=UPDATE_EXISTING_REPOS,
        preserve_local_changes=PRESERVE_LOCAL_REPO_CHANGES,
    ))
if CLONE_FLEXDC_REPO:
    repo_states.append(clone_or_update_repository(
        name="FlexDC", url=FLEXDC_REPO_URL, destination=FLEXDC_ROOT,
        branch=FLEXDC_BRANCH, force_reclone=FORCE_FRESH_CLONE,
        update_existing=UPDATE_EXISTING_REPOS,
        preserve_local_changes=PRESERVE_LOCAL_REPO_CHANGES,
    ))
if not FLEXDC_ROOT.exists():
    raise FileNotFoundError(f"FlexDC root is unavailable: {FLEXDC_ROOT}. Enable cloning or set FLEXDC_ROOT_OVERRIDE.")
REPO_MANIFEST = repository_manifest(repo_states, WORKSPACE / "repository_manifest.json")

if INSTALL_FLEXDC_REQUIREMENTS and FLEXDC_ROOT.exists():
    requirements = FLEXDC_ROOT / "requirements.txt"
    if requirements.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
if INSTALL_FLEXDC_EDITABLE and FLEXDC_ROOT.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(FLEXDC_ROOT)], check=True)

CONDOR_TRAIN_DIR = COMDER_ROOT / "am_flexdc" / "train"
GENERIC_SOURCE_INSTALL = (
    install_generic_sources_to_condor(SOURCE_DIR, CONDOR_TRAIN_DIR)
    if INSTALL_GENERIC_SOURCES_IN_CONDOR and COMDER_ROOT.exists() else {"files": []}
)

for state in repo_states:
    print(state.to_dict())
print("Generic sources installed in CONDOR train tree:", len(GENERIC_SOURCE_INSTALL.get("files", [])))


## 4. Acquire model artifact(s), prior outputs, and install runtime bundles

In [ ]:
import re, json, pandas as pd

ARTIFACTS_ROOT = WORKSPACE / "artifacts"
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

# Resolve artifact inputs using every acquisition mode from the original notebook.
inputs = []
if ARTIFACT_MODE == "existing":
    inputs = [Path(p).expanduser().resolve() for p in MODEL_ARTIFACT_INPUTS]
elif ARTIFACT_MODE == "path_zip":
    inputs = [Path(p).expanduser().resolve() for p in MODEL_ARTIFACT_INPUTS]
elif ARTIFACT_MODE in {"upload_zip", "multi_upload"}:
    inputs = upload_files_direct(
        prompt="Upload one or more complete training artifact ZIPs.",
        multiple=(ARTIFACT_MODE == "multi_upload"),
    )
elif ARTIFACT_MODE == "repo_zip":
    search_root = Path(ARTIFACT_SEARCH_ROOT).expanduser().resolve() if ARTIFACT_SEARCH_ROOT else COMDER_ROOT / "am_flexdc" / "models"
    inputs = sorted(search_root.rglob("*.zip"))
    if not inputs:
        raise FileNotFoundError(f"No artifact ZIPs under {search_root}")
else:
    raise ValueError(f"Unknown ARTIFACT_MODE: {ARTIFACT_MODE}")

inputs = [Path(p).expanduser().resolve() for p in inputs if Path(p).expanduser().exists()]
if not inputs and AUTO_UPLOAD_MODEL_ARTIFACTS:
    inputs = upload_files_direct(prompt="Upload one or more complete training artifact ZIPs.", multiple=True)
if not inputs:
    raise FileNotFoundError("No model artifact inputs")

artifact_roots = []
RUNTIME_INSTALLS = []
for index, source in enumerate(inputs):
    if source.is_dir():
        root = source
    elif source.suffix.lower() == ".zip":
        root, info = extract_zip_verified(
            source, ARTIFACTS_ROOT / f"artifact_{index}",
            clean_destination=True, reject_exact_25_mib=False,
        )
        print("Artifact ZIP:", source.name, info["size_bytes"], info["sha256"])
    else:
        raise ValueError(source)
    artifact_roots.append(root)
    RUNTIME_INSTALLS.append(install_runtime_bundle(root, FLEXDC_ROOT))

# Prior results may be a configured path or direct upload. They are restored in
# the output tree after OUTPUT_ROOT is created.
PRIOR_RESULTS_ROOT = None
prior_source = None
if PRIOR_RESULTS_INPUT:
    prior_source = Path(PRIOR_RESULTS_INPUT).expanduser().resolve()
elif AUTO_UPLOAD_PRIOR_RESULTS:
    uploaded = upload_files_direct(prompt="Upload the prior inference-result ZIP.", multiple=False)
    prior_source = uploaded[0]
if prior_source is not None:
    PRIOR_RESULTS_ROOT = (
        extract_zip_verified(prior_source, WORKSPACE / "prior_results", clean_destination=True, reject_exact_25_mib=False)[0]
        if prior_source.suffix.lower() == ".zip" else prior_source
    )

MODEL_RUNTIMES = []
for index, root in enumerate(artifact_roots):
    explicit = EXPLICIT_CHECKPOINT_PATHS[index] if index < len(EXPLICIT_CHECKPOINT_PATHS) else None
    checkpoint = select_checkpoint(root, preferred_role=PREFERRED_CHECKPOINT_ROLE, explicit_path=explicit)
    label = MODEL_LABEL_OVERRIDES[index] if index < len(MODEL_LABEL_OVERRIDES) else checkpoint.parent.name or f"model_{index+1}"
    MODEL_RUNTIMES.append({
        "label": re.sub(r"[^A-Za-z0-9._-]+", "_", str(label)).strip("_") or f"model_{index+1}",
        "root": root,
        "checkpoint": checkpoint,
    })

print(pd.DataFrame([{k: str(v) for k, v in item.items()} for item in MODEL_RUNTIMES]).to_string(index=False))


## 5. Resolve source files, compile, and run structural tests

In [ ]:
import os
for path in sorted(SOURCE_DIR.glob("*.py")): subprocess.run([sys.executable,"-m","py_compile",str(path)],check=True)
env=os.environ.copy(); env["PYTHONPATH"]=str(SOURCE_DIR)+os.pathsep+env.get("PYTHONPATH","")
for test in [SOURCE_DIR/"test_flexdc_behavior_training.py",SOURCE_DIR/"test_flexdc_profile_holdout.py"]:
    subprocess.run([sys.executable,str(test)],cwd=SOURCE_DIR,env=env,check=True)
required=[FLEXDC_ROOT/"src/peacsim/am_data_extraction_wizard.py",FLEXDC_ROOT/"configs/cluster/cluster.ini"]
missing=[p for p in required if not p.exists()]
if missing: raise FileNotFoundError(missing)
print("Inference source and FlexDC runtime checks: PASS")

## 6. Import low-level inference, optimization, validation, and presentation helpers

In [ ]:
import numpy as np, pandas as pd, torch, json, math, re, shutil
from IPython.display import HTML, display

torch.set_num_threads(max(1, int(TORCH_CPU_THREADS)))

from flexdc_behavior_inference_utilities import (
    OptimizationSettings, calculate_pr_bounds, calculate_weight_bounds,
    dataframe_for_csv, load_behavior_model, margin_calibration_table,
    optimize_candidates, predict_configuration, read_experiment_config,
    read_workload_config, resolve_safety_limits, run_flexdc_validation,
)
from flexdc_behavior_evaluation import render_styled_table
from flexdc_inference_orchestration import (
    ScenarioDefinition, aggregate_validation, build_profile_suite,
    build_workload_catalog, compare_models_on_points, restore_prior_outputs,
    run_scenario_case, run_scenario_suite, safe_tag,
)
from flexdc_presentation import build_report, write_report

def show(frame, caption=None, precision=5, max_rows=500):
    if frame is None or len(frame) == 0:
        print((caption or "Table") + ": no rows")
        return
    sample = frame.head(max_rows)
    if DISPLAY_TABLES:
        render_styled_table(sample, caption=caption, precision=precision)
    else:
        print((caption or "") + "\n" + sample.to_string(index=False))

LOADED_MODELS = []
for item in MODEL_RUNTIMES:
    loaded = load_behavior_model(item["checkpoint"], device_name="auto")
    training_config = loaded.checkpoint.get("training_config", {})
    runtime = {
        **item,
        "loaded": loaded,
        "epoch": int(loaded.checkpoint.get("epoch", -1)),
        "model_id": training_config.get("model_id"),
        "run_name": training_config.get("run_name"),
    }
    LOADED_MODELS.append(runtime)

show(pd.DataFrame([
    {k: (str(v) if k in {"root", "checkpoint"} else v) for k, v in model.items() if k != "loaded"}
    for model in LOADED_MODELS
]), "Loaded model runtimes")


## 7. Start or resume the W&B inference run

In [ ]:
run=None
if USE_WANDB and WANDB_MODE!="disabled":
    import wandb, getpass
    os.environ["WANDB_MODE"]=WANDB_MODE
    if WANDB_MODE=="online":
        try: wandb.login()
        except Exception: wandb.login(key=getpass.getpass("Paste W&B API key: "),relogin=True)
    run=wandb.init(project=WANDB_PROJECT,entity=WANDB_ENTITY,name=f"generic_inference_{safe_tag(SCENARIO)}",mode=WANDB_MODE,id=WANDB_RUN_ID_OVERRIDE,resume="allow" if WANDB_RUN_ID_OVERRIDE else None,config={"scenario":SCENARIO,"models":[m["label"] for m in LOADED_MODELS],"starts":MULTI_STARTS,"iterations":OPTIMIZATION_ITERATIONS,"simulator_seeds":SIMULATOR_SEEDS})
    print("W&B run:",run.url if getattr(run,"url",None) else run.id)
else: print("W&B disabled")

## 8. Build the workload catalog and select a preset or custom scenario

In [ ]:
workload_dir = FLEXDC_ROOT / "configs/workload/j2_pairwise"
WORKLOAD_CATALOG = build_workload_catalog(workload_dir)

if SCENARIO == "CUSTOM":
    if CUSTOM_WORKLOAD_CONFIG is None:
        raise ValueError("CUSTOM_WORKLOAD_CONFIG required")
    WORKLOAD_CONFIG = Path(CUSTOM_WORKLOAD_CONFIG).expanduser()
    if not WORKLOAD_CONFIG.is_absolute():
        WORKLOAD_CONFIG = FLEXDC_ROOT / WORKLOAD_CONFIG
else:
    key = SCENARIO[:-4] if SCENARIO.endswith(".ini") else SCENARIO
    if key not in WORKLOAD_CATALOG:
        print("Available J2 scenarios:", *WORKLOAD_CATALOG, sep="\n - ")
        raise KeyError(key)
    WORKLOAD_CONFIG = WORKLOAD_CATALOG[key]

EXPERIMENT_CONFIG = Path(CUSTOM_EXPERIMENT_CONFIG).expanduser()
if not EXPERIMENT_CONFIG.is_absolute():
    EXPERIMENT_CONFIG = FLEXDC_ROOT / EXPERIMENT_CONFIG
GRADIENT_CONFIG = FLEXDC_ROOT / "configs/gradient_descent/gradient_descent_j2_pairwise_rsr.ini"
CLUSTER_CONFIG = FLEXDC_ROOT / "configs/cluster/cluster.ini"

required = [WORKLOAD_CONFIG, EXPERIMENT_CONFIG]
if RUN_FLEXDC_VALIDATION or RUN_FIXED_PROFILE_SUITE or RUN_ALL_WORKLOADS or RUN_LEGACY_CUSTOM_ROUNDS:
    required.extend([GRADIENT_CONFIG, CLUSTER_CONFIG])
missing = [path for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing runtime files:\n" + "\n".join(f" - {path}" for path in missing))

WORKLOAD = read_workload_config(WORKLOAD_CONFIG)
EXPERIMENT = read_experiment_config(EXPERIMENT_CONFIG, server_count_override=SERVER_COUNT, utilization_override=UTILIZATION)
BOUNDS = calculate_pr_bounds(
    WORKLOAD, pbar_lower_factor=PBAR_LOWER_FACTOR,
    pbar_upper_factor=PBAR_UPPER_FACTOR, pr_upper_factor=PR_UPPER_FACTOR,
    r_lower_kw_per_server=R_LOWER_KW_PER_SERVER,
)

if START_WEIGHTS is None:
    START_WEIGHTS = [1.0 / WORKLOAD.job_count] * WORKLOAD.job_count
if START_POINT_MODE == "absolute":
    if START_PBAR is None or START_R is None:
        raise ValueError("absolute mode requires START_PBAR and START_R")
elif START_POINT_MODE == "ratios":
    pspan = BOUNDS.pbar_upper_kw_per_server - BOUNDS.pbar_lower_kw_per_server
    START_PBAR = BOUNDS.pbar_lower_kw_per_server + float(START_PBAR_RATIO) * pspan
    START_R = float(START_R_RATIO) * START_PBAR
elif START_POINT_MODE == "midpoint":
    START_PBAR = (BOUNDS.pbar_lower_kw_per_server + BOUNDS.pbar_upper_kw_per_server) / 2 if START_PBAR is None else START_PBAR
    START_R = min(0.30 * START_PBAR, BOUNDS.pr_upper_kw_per_server - START_PBAR) if START_R is None else START_R
else:
    raise ValueError(START_POINT_MODE)

print("Workload:", WORKLOAD_CONFIG)
print("Experiment:", EXPERIMENT_CONFIG)
print("Jobs:", WORKLOAD.job_names)
print("P/R bounds:", BOUNDS.to_dict())
print("Anchor:", START_PBAR, START_R, START_WEIGHTS)


## 9. Inspect model safety limits, legal weight domain, and optional calibration

In [ ]:
SAFETY_BY_MODEL={}
rows=[]
for runtime in LOADED_MODELS:
    safety=resolve_safety_limits(runtime["loaded"].constants,tracking_margin=TRACKING_MARGIN,qos_margin=QOS_MARGIN)
    SAFETY_BY_MODEL[runtime["label"]]=safety
    wb=calculate_weight_bounds(WORKLOAD.job_count,EXPERIMENT.server_count)
    rows.append({"Model":runtime["label"],"Epoch":runtime["epoch"],"Exact Tracking":safety.exact_tracking_threshold,"Selection Tracking":safety.selection_tracking_limit,"Exact QoS":safety.exact_qos_threshold,"Selection QoS":safety.selection_qos_limit,"Wizard Weight Min":wb.final_lower,"Wizard Weight Max":wb.final_upper})
show(pd.DataFrame(rows),"Safety and weight domain")
CALIBRATION_TABLES={}
if RUN_SAFETY_CALIBRATION:
    for runtime in LOADED_MODELS:
        predictions=sorted(runtime["root"].rglob("*validation_predictions.csv"))+sorted(runtime["root"].rglob("*test_predictions.csv"))
        if predictions:
            table=margin_calibration_table(predictions[0]); CALIBRATION_TABLES[runtime["label"]]=table; show(table.head(20),f"Safety calibration — {runtime['label']}")
        else: print("No prediction CSV for calibration:",runtime["label"])

## 10. Predict one point with every loaded model

In [ ]:
OUTPUT_ROOT = WORKSPACE / "inference_results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESTORED_PRIOR_FILES = restore_prior_outputs(PRIOR_RESULTS_ROOT, OUTPUT_ROOT) if RESUME_EXISTING_RESULTS else []
print("Recovered prior output files:", len(RESTORED_PRIOR_FILES))

PREDICT_ROWS = []
PREDICT_PER_JOB = []
if RUN_PREDICT_ONE:
    for runtime in LOADED_MODELS:
        summary, per_job = predict_configuration(
            runtime["loaded"], workload=WORKLOAD, experiment=EXPERIMENT,
            pbar_kw_per_server=START_PBAR, r_kw_per_server=START_R,
            weights=START_WEIGHTS, safety=SAFETY_BY_MODEL[runtime["label"]],
        )
        PREDICT_ROWS.append({"Model": runtime["label"], **summary})
        per_job = per_job.copy()
        per_job.insert(0, "Model", runtime["label"])
        PREDICT_PER_JOB.append(per_job)

PREDICT_ONE_SUMMARY = pd.DataFrame(PREDICT_ROWS)
PREDICT_ONE_PER_JOB = pd.concat(PREDICT_PER_JOB, ignore_index=True) if PREDICT_PER_JOB else pd.DataFrame()
show(PREDICT_ONE_SUMMARY, "Predict One — model summaries")
show(PREDICT_ONE_PER_JOB, "Predict One — per-job QoS")
dataframe_for_csv(PREDICT_ONE_SUMMARY).to_csv(OUTPUT_ROOT / "predict_one_summary.csv", index=False)
dataframe_for_csv(PREDICT_ONE_PER_JOB).to_csv(OUTPUT_ROOT / "predict_one_per_job.csv", index=False)


## 11. Standalone Predict One CLI command

In [ ]:
PREDICT_CLI=[sys.executable,str(SOURCE_DIR/"flexdc_behavior_predict_one.py"),"--checkpoint",str(LOADED_MODELS[0]["checkpoint"]),"--workload-config",str(WORKLOAD_CONFIG),"--experiment-config",str(EXPERIMENT_CONFIG),"--pbar",str(START_PBAR),"--r",str(START_R),"--weights",",".join(map(str,START_WEIGHTS)),"--server-count",str(SERVER_COUNT),"--utilization",str(UTILIZATION),"--out-dir",str(OUTPUT_ROOT/"predict_one_cli"),"--run-name","predict_one"]
print("Standalone command:\n"," ".join(PREDICT_CLI))
# Uncomment to execute the standalone path as an additional integration test:
# subprocess.run(PREDICT_CLI,check=True)

## 12. Multi-start constrained optimization with shared deterministic starts

In [ ]:
BASE_OPTIMIZATION_SETTINGS = OptimizationSettings(
    starts=MULTI_STARTS,
    iterations=OPTIMIZATION_ITERATIONS,
    learning_rate=OPTIMIZATION_LR,
    minimum_learning_rate=OPTIMIZATION_MIN_LR,
    mode=OPTIMIZATION_MODE,
    tracking_penalty=TRACKING_PENALTY,
    qos_penalty=QOS_PENALTY,
    penalty_ramp_fraction=PENALTY_RAMP_FRACTION,
    top_k=TOP_K,
    candidate_distance=CANDIDATE_DISTANCE,
    random_seed=RANDOM_SEED,
    near_equal_start_fraction=NEAR_EQUAL_START_FRACTION,
    high_p_low_r_start_fraction=HIGH_P_LOW_R_START_FRACTION,
    enforce_flexdc_weight_bounds=ENFORCE_FLEXDC_WEIGHT_BOUNDS,
    weight_min_fraction_of_equal=WEIGHT_MIN_FRACTION_OF_EQUAL,
    weight_max_multiple_of_equal=WEIGHT_MAX_MULTIPLE_OF_EQUAL,
    weight_min=WEIGHT_MIN if WORKLOAD.job_count == 2 else None,
    weight_max=WEIGHT_MAX if WORKLOAD.job_count == 2 else None,
    r_over_p_max=R_OVER_P_MAX,
    pbar_min_override=PBAR_MIN_OVERRIDE,
    pbar_max_override=PBAR_MAX_OVERRIDE,
    r_min_override=R_MIN_OVERRIDE,
    r_max_override=R_MAX_OVERRIDE,
    log_every=LOG_EVERY,
)

OPTIMIZATION_RESULTS = {}
if RUN_OPTIMIZE_ONE:
    for runtime in LOADED_MODELS:
        case_dir = OUTPUT_ROOT / "optimize_one" / runtime["label"]
        case_dir.mkdir(parents=True, exist_ok=True)
        candidates, top_k, trajectory = optimize_candidates(
            runtime["loaded"], workload=WORKLOAD, experiment=EXPERIMENT,
            bounds=BOUNDS, safety=SAFETY_BY_MODEL[runtime["label"]],
            settings=BASE_OPTIMIZATION_SETTINGS,
            initial_pbar=START_PBAR, initial_reserve=START_R,
            initial_weights=START_WEIGHTS,
        )
        OPTIMIZATION_RESULTS[runtime["label"]] = {
            "settings": BASE_OPTIMIZATION_SETTINGS, "candidates": candidates,
            "top_k": top_k, "trajectory": trajectory, "dir": case_dir,
        }
        dataframe_for_csv(candidates).to_csv(case_dir / "all_starts.csv", index=False)
        dataframe_for_csv(top_k).to_csv(case_dir / "top_k.csv", index=False)
        dataframe_for_csv(trajectory).to_csv(case_dir / "trajectory.csv", index=False)
        show(top_k, f"Top-{TOP_K} — {runtime['label']}")


## 13. Plot and save optimization trajectories

In [ ]:
import matplotlib.pyplot as plt
TRAJECTORY_PLOTS=[]
for label,result in OPTIMIZATION_RESULTS.items():
    trajectory=result["trajectory"]
    if trajectory.empty: continue
    for column in ["Best_Predicted_Objective","Safety_Feasible_Starts","Median_P90","Median_Max_Pj"]:
        if column not in trajectory: continue
        fig=plt.figure(figsize=(8,4)); plt.plot(trajectory["Iteration"],trajectory[column]); plt.xlabel("Iteration"); plt.ylabel(column); plt.title(f"{label}: {column}"); plt.grid(True,alpha=.3); plt.tight_layout(); path=result["dir"]/f"trajectory_{safe_tag(column)}.png"; fig.savefig(path,dpi=160,bbox_inches="tight"); TRAJECTORY_PLOTS.append(path); plt.show(); plt.close(fig)

## 14. Validate the starting point and top-k candidates in real FlexDC over multiple seeds

In [ ]:
import configparser
VALIDATION_RESULTS={}

def experiment_for_seed(base_path,seed,target_dir):
    parser=configparser.ConfigParser(interpolation=None); parser.read(base_path)
    if not parser.has_section("system"): raise KeyError("[system]")
    parser.set("system","random_seed",str(seed))
    target=target_dir/f"experiment_seed_{seed}.ini"
    with target.open("w") as handle: parser.write(handle)
    return target

if RUN_FLEXDC_VALIDATION:
    for runtime in LOADED_MODELS:
        rows=[]; per_jobs=[]; result=OPTIMIZATION_RESULTS.get(runtime["label"])
        candidates=[]
        if VALIDATE_STARTING_POINT: candidates.append({"Candidate_Type":"Anchor","Candidate_Rank":0,"Pbar_kw_per_server":START_PBAR,"R_kw_per_server":START_R,"weights":START_WEIGHTS,**(PREDICT_ONE_SUMMARY[PREDICT_ONE_SUMMARY["Model"]==runtime["label"]].iloc[0].to_dict() if len(PREDICT_ONE_SUMMARY) else {})})
        if VALIDATE_TOP_K and result is not None:
            for _,row in result["top_k"].iterrows(): candidates.append({"Candidate_Type":"Optimized","Candidate_Rank":int(row["Candidate_Rank"]),**row.to_dict()})
        validation_dir=OUTPUT_ROOT/"flexdc_validation"/runtime["label"]; validation_dir.mkdir(parents=True,exist_ok=True)
        for candidate in candidates:
            for seed in SIMULATOR_SEEDS:
                exp_seed=experiment_for_seed(EXPERIMENT_CONFIG,seed,validation_dir)
                actual,jobs=run_flexdc_validation(python_executable=sys.executable,flexdc_root=FLEXDC_ROOT,gradient_config=GRADIENT_CONFIG,experiment_config=exp_seed,cluster_config=CLUSTER_CONFIG,workload_config=WORKLOAD_CONFIG,output_label=f"{safe_tag(runtime['label'])}_{safe_tag(SCENARIO)}_{candidate['Candidate_Type']}_r{candidate['Candidate_Rank']}_s{seed}",pbar_kw_per_server=float(candidate["Pbar_kw_per_server"]),r_kw_per_server=float(candidate["R_kw_per_server"]),weights=candidate["weights"],utilization=UTILIZATION,constants=runtime["loaded"].constants,policy_name="AQA",node_count_control=True,timeout_seconds=VALIDATION_TIMEOUT_SECONDS,dry_run=False)
                rows.append({"Model":runtime["label"],"Seed":seed,**candidate,**actual})
                if len(jobs): jobs=jobs.copy(); jobs.insert(0,"Model",runtime["label"]); jobs.insert(1,"Seed",seed); jobs.insert(2,"Candidate_Type",candidate["Candidate_Type"]); jobs.insert(3,"Candidate_Rank",candidate["Candidate_Rank"]); per_jobs.append(jobs)
        table=pd.DataFrame(rows); jobs_table=pd.concat(per_jobs,ignore_index=True) if per_jobs else pd.DataFrame(); VALIDATION_RESULTS[runtime["label"]]={"summary":table,"per_job":jobs_table}; dataframe_for_csv(table).to_csv(validation_dir/"predicted_vs_actual.csv",index=False); dataframe_for_csv(jobs_table).to_csv(validation_dir/"per_job_qos.csv",index=False); show(table,f"Predicted versus actual FlexDC — {runtime['label']}")
else: print("Real FlexDC validation disabled.")

## 15. Aggregate predicted-versus-actual and pass/fail tables

In [ ]:
all_validation=pd.concat([value["summary"] for value in VALIDATION_RESULTS.values() if len(value["summary"])],ignore_index=True) if VALIDATION_RESULTS else pd.DataFrame()
if len(all_validation):
    group_cols=[c for c in ["Model","Candidate_Type","Candidate_Rank"] if c in all_validation]
    agg=all_validation.groupby(group_cols,dropna=False).agg(Runs=("Seed","size"),Seeds_Passed=("Actual_Both_Pass","sum"),Worst_P90=("Actual_P90_Tracking","max"),Worst_Max_Pj=("Actual_Max_Pj","max"),Mean_Actual_Objective=("Actual_Full_Objective","mean")).reset_index()
    agg["All_Seeds_Pass"]=agg["Seeds_Passed"]==agg["Runs"]
    show(agg,"Simulator validation aggregate")
    agg.to_csv(OUTPUT_ROOT/"simulator_validation_aggregate.csv",index=False)

## 16. Fixed seen / one-held / two-held benchmark suite

In [ ]:
FIXED_SUITE_RESULTS = []
FIXED_SUITE_SUMMARY = pd.DataFrame()
if RUN_FIXED_PROFILE_SUITE:
    primary_model_id = LOADED_MODELS[0].get("model_id")
    selected = build_profile_suite(
        WORKLOAD_CATALOG, model_id=primary_model_id,
        per_benchmark=FIXED_SUITE_PER_BENCHMARK,
    )
    show(selected, "Fixed seen / one-held / two-held profile suite")
    fixed_cases = [
        ScenarioDefinition(
            case_id=f"fixed_{safe_tag(row.Benchmark_Type)}_{safe_tag(row.Workload_Name)}_u{UTILIZATION}",
            workload_config=str(row.Path), experiment_config=str(EXPERIMENT_CONFIG),
            server_count=SERVER_COUNT, utilization=UTILIZATION,
            initial_pbar=START_PBAR, initial_r=START_R,
            initial_weights=tuple([0.5, 0.5]), category=row.Benchmark_Type,
        )
        for row in selected.itertuples(index=False)
    ]
    FIXED_SUITE_RESULTS, FIXED_SUITE_SUMMARY = run_scenario_suite(
        model_runtimes=LOADED_MODELS, cases=fixed_cases,
        output_root=OUTPUT_ROOT / "fixed_profile_suite",
        settings=BASE_OPTIMIZATION_SETTINGS,
        tracking_margin=TRACKING_MARGIN, qos_margin=QOS_MARGIN,
        run_flexdc=RUN_FLEXDC_VALIDATION,
        validate_anchor=VALIDATE_STARTING_POINT, validate_top_k=VALIDATE_TOP_K,
        simulator_seeds=SIMULATOR_SEEDS, flexdc_root=FLEXDC_ROOT,
        gradient_config=GRADIENT_CONFIG, cluster_config=CLUSTER_CONFIG,
        validation_timeout_seconds=VALIDATION_TIMEOUT_SECONDS,
        resume=RESUME_EXISTING_RESULTS, dry_run_flexdc=DRY_RUN_FLEXDC,
    )
    show(FIXED_SUITE_SUMMARY, "Fixed profile-suite summary")


## 17. Optional custom legacy rounds

In [ ]:
LEGACY_ROUND_RESULTS = []
LEGACY_ROUND_SUMMARY = pd.DataFrame()
if RUN_LEGACY_CUSTOM_ROUNDS:
    if not LEGACY_CUSTOM_ROUNDS:
        raise ValueError("Populate LEGACY_CUSTOM_ROUNDS with scenario dictionaries")
    legacy_cases = [ScenarioDefinition.from_mapping(case) for case in LEGACY_CUSTOM_ROUNDS]
    show(pd.DataFrame([case.to_dict() for case in legacy_cases]), "Configured custom/legacy rounds")
    LEGACY_ROUND_RESULTS, LEGACY_ROUND_SUMMARY = run_scenario_suite(
        model_runtimes=LOADED_MODELS, cases=legacy_cases,
        output_root=OUTPUT_ROOT / "custom_rounds",
        settings=BASE_OPTIMIZATION_SETTINGS,
        tracking_margin=TRACKING_MARGIN, qos_margin=QOS_MARGIN,
        run_flexdc=RUN_FLEXDC_VALIDATION,
        validate_anchor=VALIDATE_STARTING_POINT, validate_top_k=VALIDATE_TOP_K,
        simulator_seeds=SIMULATOR_SEEDS, flexdc_root=FLEXDC_ROOT,
        gradient_config=GRADIENT_CONFIG, cluster_config=CLUSTER_CONFIG,
        validation_timeout_seconds=VALIDATION_TIMEOUT_SECONDS,
        resume=RESUME_EXISTING_RESULTS, dry_run_flexdc=DRY_RUN_FLEXDC,
    )
    show(LEGACY_ROUND_SUMMARY, "Custom/legacy-round summary")


## 18. Resumable all-workload optimization and validation

In [ ]:
ALL_WORKLOAD_RESULTS = []
ALL_WORKLOAD_SUMMARY = pd.DataFrame()
ALL_WORKLOAD_ERRORS = []
if RUN_ALL_WORKLOADS:
    names = sorted(WORKLOAD_CATALOG)
    if ALL_WORKLOAD_NAME_FILTER:
        names = [name for name in names if re.search(ALL_WORKLOAD_NAME_FILTER, name, re.I)]
    if ALL_WORKLOAD_CATEGORY_FILTER:
        names = [name for name in names if ALL_WORKLOAD_CATEGORY_FILTER.lower() in name.lower()]
    if ALL_WORKLOAD_MAX_CASES is not None:
        names = names[:int(ALL_WORKLOAD_MAX_CASES)]
    cases = [
        ScenarioDefinition(
            case_id=f"all_{safe_tag(name)}_u{UTILIZATION}",
            workload_config=str(WORKLOAD_CATALOG[name]),
            experiment_config=str(EXPERIMENT_CONFIG),
            server_count=SERVER_COUNT, utilization=UTILIZATION,
            initial_weights=(0.5, 0.5),
        )
        for name in names
    ]
    # Run one case at a time so optional stop-on-error behavior matches the old batch notebook.
    summaries = []
    for case in cases:
        try:
            results, summary = run_scenario_suite(
                model_runtimes=LOADED_MODELS, cases=[case],
                output_root=OUTPUT_ROOT / "all_workloads",
                settings=BASE_OPTIMIZATION_SETTINGS,
                tracking_margin=TRACKING_MARGIN, qos_margin=QOS_MARGIN,
                run_flexdc=RUN_FLEXDC_VALIDATION and ALL_WORKLOAD_VALIDATE_TOP_K,
                validate_anchor=VALIDATE_STARTING_POINT,
                validate_top_k=ALL_WORKLOAD_VALIDATE_TOP_K,
                simulator_seeds=ALL_WORKLOAD_SEEDS,
                flexdc_root=FLEXDC_ROOT, gradient_config=GRADIENT_CONFIG,
                cluster_config=CLUSTER_CONFIG,
                validation_timeout_seconds=VALIDATION_TIMEOUT_SECONDS,
                resume=RESUME_EXISTING_RESULTS, dry_run_flexdc=DRY_RUN_FLEXDC,
            )
            ALL_WORKLOAD_RESULTS.extend(results)
            summaries.append(summary)
        except Exception as exc:
            ALL_WORKLOAD_ERRORS.append({"Case_ID": case.case_id, "Error": repr(exc)})
            print("FAILED:", case.case_id, repr(exc))
            if ALL_WORKLOAD_STOP_ON_ERROR:
                raise
    ALL_WORKLOAD_SUMMARY = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
    show(ALL_WORKLOAD_SUMMARY, "All-workload optimization/validation summary")
    if ALL_WORKLOAD_ERRORS:
        show(pd.DataFrame(ALL_WORKLOAD_ERRORS), "All-workload errors")


## 19. Identical-point and shared-start multi-model comparison

In [ ]:
MULTI_MODEL_COMPARISON = pd.DataFrame()
SHARED_START_OPTIMIZATION_SUMMARY = pd.DataFrame()
REFERENCE_COMPARISON = pd.DataFrame()

# Identical-point predictions: anchor plus every primary-model top-k point.
if RUN_MULTI_MODEL_IDENTICAL_POINT_COMPARISON:
    points = [{"Point": "Anchor", "pbar": START_PBAR, "r": START_R, "weights": START_WEIGHTS}]
    primary_top = OPTIMIZATION_RESULTS.get(LOADED_MODELS[0]["label"], {}).get("top_k", pd.DataFrame())
    for _, row in primary_top.iterrows():
        points.append({
            "Point": f"Primary Rank {int(row['Candidate_Rank'])}",
            "pbar": float(row["Pbar_kw_per_server"]),
            "r": float(row["R_kw_per_server"]),
            "weights": row["weights"],
        })
    MULTI_MODEL_COMPARISON = compare_models_on_points(
        model_runtimes=LOADED_MODELS, workload_config=WORKLOAD_CONFIG,
        experiment_config=EXPERIMENT_CONFIG, points=points,
        server_count=SERVER_COUNT, utilization=UTILIZATION,
        tracking_margin=TRACKING_MARGIN, qos_margin=QOS_MARGIN,
    )
    show(MULTI_MODEL_COMPARISON, "Models scored on identical points")
    dataframe_for_csv(MULTI_MODEL_COMPARISON).to_csv(OUTPUT_ROOT / "identical_point_model_comparison.csv", index=False)

# Shared deterministic starts are already guaranteed by the same settings/random seed.
if RUN_MULTI_MODEL_SHARED_START_OPTIMIZATION and len(LOADED_MODELS) >= 2:
    rows = []
    for runtime in LOADED_MODELS:
        result = OPTIMIZATION_RESULTS.get(runtime["label"])
        if result is None:
            continue
        top = result["top_k"]
        rows.append({
            "Model": runtime["label"],
            "Starts": len(result["candidates"]),
            "Safety_Feasible_Starts": int(result["candidates"]["Safety_Both_Pass"].sum()),
            "Top_K": len(top),
            "Best_Predicted_Objective": float(top.iloc[0]["Predicted_Full_Objective"]) if len(top) else np.nan,
        })
    SHARED_START_OPTIMIZATION_SUMMARY = pd.DataFrame(rows)
    show(SHARED_START_OPTIMIZATION_SUMMARY, "Shared-start multi-model optimization")

# Optional external/Fatih/reference points. Required columns: Point, Pbar_kw_per_server,
# R_kw_per_server, weights. Any actual columns are preserved for presentation.
if REFERENCE_POINTS_CSV:
    reference_path = Path(REFERENCE_POINTS_CSV).expanduser().resolve()
    references = pd.read_csv(reference_path, low_memory=False)
    reference_points = [
        {"Point": row["Point"], "pbar": row["Pbar_kw_per_server"], "r": row["R_kw_per_server"], "weights": row["weights"]}
        for _, row in references.iterrows()
    ]
    predicted_reference = compare_models_on_points(
        model_runtimes=LOADED_MODELS, workload_config=WORKLOAD_CONFIG,
        experiment_config=EXPERIMENT_CONFIG, points=reference_points,
        server_count=SERVER_COUNT, utilization=UTILIZATION,
        tracking_margin=TRACKING_MARGIN, qos_margin=QOS_MARGIN,
    )
    REFERENCE_COMPARISON = predicted_reference.merge(references, on="Point", how="left", suffixes=("_Predicted", "_Reference"))
    show(REFERENCE_COMPARISON, "Model predictions on external reference points")
    dataframe_for_csv(REFERENCE_COMPARISON).to_csv(OUTPUT_ROOT / "external_reference_comparison.csv", index=False)


## 20. Presentation-ready HTML tables

In [ ]:
PRESENTATION_FILES = []
report_tables = {
    "Predict One": PREDICT_ONE_SUMMARY,
    "Predict One per-job QoS": PREDICT_ONE_PER_JOB,
    "Simulator validation": all_validation,
    "Simulator validation aggregate": aggregate_validation(all_validation),
    "Fixed profile suite": FIXED_SUITE_SUMMARY,
    "Custom rounds": LEGACY_ROUND_SUMMARY,
    "All workloads": ALL_WORKLOAD_SUMMARY,
    "Shared-start model summary": SHARED_START_OPTIMIZATION_SUMMARY,
    "External references": REFERENCE_COMPARISON,
}
for label, result in OPTIMIZATION_RESULTS.items():
    report_tables[f"Top-{TOP_K}: {label}"] = result["top_k"]

report_html = build_report(
    title="CONDOR–FlexDC generic inference and simulator validation",
    subtitle=f"Scenario {SCENARIO}; checkpoint role {PREFERRED_CHECKPOINT_ROLE}",
    tables={name: frame for name, frame in report_tables.items() if frame is not None and len(frame)},
    comparisons=[("Identical-point multi-model comparison", MULTI_MODEL_COMPARISON)] if len(MULTI_MODEL_COMPARISON) else [],
    notes=[
        "Predicted feasibility is not treated as simulator validation.",
        "All real-validation tables show seed-level FlexDC outputs and an all-seeds aggregate.",
    ],
)
PRESENTATION_HTML = write_report(OUTPUT_ROOT / "presentation_ready_results.html", report_html)
PRESENTATION_FILES.append(PRESENTATION_HTML)
if DISPLAY_TABLES:
    display(HTML(report_html))


## 21. W&B logging, including recovered and batch results

In [ ]:
if run is not None:
    wandb_tables = {
        "predict_one": PREDICT_ONE_SUMMARY,
        "simulator_validation": all_validation,
        "fixed_profile_suite": FIXED_SUITE_SUMMARY,
        "custom_rounds": LEGACY_ROUND_SUMMARY,
        "all_workloads": ALL_WORKLOAD_SUMMARY,
        "multi_model_comparison": MULTI_MODEL_COMPARISON,
        "shared_start_optimization": SHARED_START_OPTIMIZATION_SUMMARY,
        "external_reference_comparison": REFERENCE_COMPARISON,
    }
    for name, frame in wandb_tables.items():
        if frame is not None and len(frame):
            run.log({name: wandb.Table(dataframe=dataframe_for_csv(frame))})
    for label, result in OPTIMIZATION_RESULTS.items():
        if len(result["top_k"]):
            run.log({f"top_k/{label}": wandb.Table(dataframe=dataframe_for_csv(result["top_k"]))})
    run.summary["models_loaded"] = len(LOADED_MODELS)
    run.summary["actual_validations"] = int(len(all_validation))
    run.summary["fixed_suite_cases"] = int(len(FIXED_SUITE_SUMMARY))
    run.summary["custom_round_cases"] = int(len(LEGACY_ROUND_SUMMARY))
    run.summary["all_workloads_completed"] = int(len(ALL_WORKLOAD_SUMMARY))
    run.summary["all_workload_errors"] = int(len(ALL_WORKLOAD_ERRORS))
    run.summary["recovered_prior_files"] = int(len(RESTORED_PRIOR_FILES))


## 22. Full standalone orchestrator command

In [ ]:
primary = LOADED_MODELS[0]
ORCHESTRATOR_COMMAND = [
    sys.executable, str(SOURCE_DIR / "flexdc_behavior_end_to_end_eval.py"),
    "--checkpoint", str(primary["checkpoint"]),
    "--workload-config", str(WORKLOAD_CONFIG),
    "--experiment-config", str(EXPERIMENT_CONFIG),
    "--server-count", str(SERVER_COUNT), "--utilization", str(UTILIZATION),
    "--starts", str(MULTI_STARTS), "--iterations", str(OPTIMIZATION_ITERATIONS),
    "--learning-rate", str(OPTIMIZATION_LR), "--minimum-learning-rate", str(OPTIMIZATION_MIN_LR),
    "--tracking-penalty", str(TRACKING_PENALTY), "--qos-penalty", str(QOS_PENALTY),
    "--penalty-ramp-fraction", str(PENALTY_RAMP_FRACTION),
    "--top-k", str(TOP_K), "--candidate-distance", str(CANDIDATE_DISTANCE),
    "--random-seed", str(RANDOM_SEED), "--r-over-p-max", str(R_OVER_P_MAX),
    "--weight-min", str(WEIGHT_MIN), "--weight-max", str(WEIGHT_MAX),
    "--out-dir", str(OUTPUT_ROOT / "full_orchestrator"),
    "--run-name", safe_tag(SCENARIO),
]
if RUN_FLEXDC_VALIDATION:
    ORCHESTRATOR_COMMAND += [
        "--run-flexdc-validation", "--flexdc-root", str(FLEXDC_ROOT),
        "--gradient-config", str(GRADIENT_CONFIG),
        "--cluster-config", str(CLUSTER_CONFIG),
        "--validation-timeout", str(VALIDATION_TIMEOUT_SECONDS),
    ]
print("Full orchestrator command:\n", " ".join(ORCHESTRATOR_COMMAND))
if RUN_FULL_ORCHESTRATOR_COMMAND:
    subprocess.run(ORCHESTRATOR_COMMAND, check=True)


## 23. Package outputs, log artifact, finish W&B, and optionally download

In [ ]:
RUN_MANIFEST = write_json(OUTPUT_ROOT / "inference_run_manifest.json", {
    "scenario": SCENARIO,
    "models": [
        {"label": model["label"], "checkpoint": str(model["checkpoint"]), "epoch": model["epoch"], "model_id": model.get("model_id")}
        for model in LOADED_MODELS
    ],
    "repositories": [state.to_dict() for state in repo_states],
    "generic_source_install": GENERIC_SOURCE_INSTALL,
    "runtime_installs": RUNTIME_INSTALLS,
    "restored_prior_files": RESTORED_PRIOR_FILES,
    "settings": {
        "starts": MULTI_STARTS, "iterations": OPTIMIZATION_ITERATIONS,
        "top_k": TOP_K, "seeds": SIMULATOR_SEEDS,
        "run_flexdc_validation": RUN_FLEXDC_VALIDATION,
        "fixed_suite": RUN_FIXED_PROFILE_SUITE,
        "custom_rounds": RUN_LEGACY_CUSTOM_ROUNDS,
        "all_workloads": RUN_ALL_WORKLOADS,
    },
})
FINAL_ZIP = package_paths(
    OUTPUT_ROOT / f"generic_inference_{safe_tag(SCENARIO)}_complete_outputs.zip",
    [OUTPUT_ROOT, SOURCE_DIR, REPO_MANIFEST, WORKLOAD_CONFIG, EXPERIMENT_CONFIG, GRADIENT_CONFIG, CLUSTER_CONFIG],
    root=WORKSPACE,
)
print("Final ZIP:", FINAL_ZIP)
print("Size MB:", FINAL_ZIP.stat().st_size / 1024**2)
print("SHA-256:", sha256_file(FINAL_ZIP))
if run is not None:
    artifact = wandb.Artifact(name=f"generic_inference_{safe_tag(SCENARIO)}", type="inference-run")
    artifact.add_file(str(FINAL_ZIP))
    run.log_artifact(artifact)
    run.finish()
download_if_colab(FINAL_ZIP, enabled=AUTO_DOWNLOAD_FINAL_ZIP)
